# Frontend HTML para etiquetado humano

**Objetivo:** crear una interfaz local para etiquetar chunks de transcripciones con categorias de moderacion de contenido.

El frontend se ejecuta como archivo HTML. No usa servidor, API ni base de datos externa. Los chunks se incrustan dentro del propio HTML (bloque `embeddedChunks`) y el anotador exporta únicamente los registros etiquetados.

In [12]:
from pathlib import Path
import itertools
import json
import pandas as pd
from collections import Counter

ROOT         = Path('..').resolve()
FRONTEND_DIR = ROOT / 'Cuadernos' / 'frontend'
PROCESSED_DIR = ROOT / 'datos' / 'processed'
LABELED_DIR  = ROOT / 'datos' / 'etiquetado'

for d in [FRONTEND_DIR, LABELED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Frontend   :', FRONTEND_DIR)
print('Processed  :', PROCESSED_DIR)
print('Etiquetado :', LABELED_DIR)


Frontend   : D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\Cuadernos\frontend
Processed  : D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\processed
Etiquetado : D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado


## 1. Taxonomia de etiquetado (fundamentada en expertos peruanos)

La taxonomia es **multi-etiqueta** y no mutuamente excluyente, salvo `seguro` vs. etiquetas de dano.

**Base academica:**
- Portocarrero (2009); Vich (2018): racismo negado — opera disfrazado de criterio de educacion o cultura.
- Zavala & Zariquiey (2007); Zavala & Back (2017): racismo encubierto y racismo linguistico.
- Almeida & Zavala (2022): motoseo y ortografia andina como discriminacion en redes sociales.
- Callirgos (1993); Branez Medina (2012): clasismo y racismo como categorias inseparables en Peru.
- Monge-Olivarria & Guerra-Corrales (2023): feminizacion como insulto en Twitter peruano.
- Thakur / CDT (2025): falla de sistemas automaticos con espanol andino y quechua mezclado.

**Proceso de decision:** 7 pasos con preguntas clave por subcategoria y 13 ejemplos anotados disponibles en:
`modelos/skills/clasificacion_moderacion_peru.md`

In [13]:
# Taxonomia peruana contextualizada — multi-etiqueta, no mutuamente excluyente
LABEL_SECTIONS = {
    'SEGURO': [
        'seguro',                    # informativo, descriptivo o humor sin ataque
        'seguro_ironia_marcada',     # parodia cuyo blanco NO es un grupo humano
    ],
    'RACISMO_DISCRIMINACION': [
        'racismo_etnico_explicito',  # serrano, cholo, negro, indio derogatorio (Callirgos, 1993)
        'racismo_linguistico',       # burla del acento andino, motoseo (Almeida & Zavala, 2022)
        'clasismo_racial',           # amixer, huachafa, chusma (Branez Medina, 2012)
        'discriminacion_regional',   # Lima vs. provincias, centralismo discriminatorio
        'racismo_encubierto',        # criterio de "cultura/educacion" para segregar (Zavala & Zariquiey, 2007)
    ],
    'ACOSO': [
        'misoginia_acoso_genero',    # insultos por ser mujer, feminizacion como insulto (Monge-Olivarria, 2023)
        'homofobia_transfobia',      # insultos/amenazas contra LGBTQ+
        'acoso_personal',            # ataque a persona identificable, doxeo
        'amenaza_directa',           # expresion explicita de intencion de dano
    ],
    'CONTENIDO_SEXUAL': [
        'sexual_explicito',          # descripcion grafica sin proposito informativo
        'sexual_cosificacion',       # sexualizacion de personas como objetos
        'sexual_no_consensual',      # revenge porn, grabacion sin consentimiento
    ],
}

# Flags transversales: se suman a categorias de dano, nunca las reemplazan
FLAGS = [
    'ironia_ambigua',       # no se distingue ironia critica de dano genuino (Vich, 2018)
    'humor_encubridor',     # el hablante usa humor para negar el dano (Branez Medina, 2012)
    'contexto_necesario',   # chunk aislado insuficiente; requiere ver el video (Thakur / CDT, 2025)
]

# Lista plana de etiquetas (sin flags) — usada por el frontend HTML
labels = [etiqueta for seccion in LABEL_SECTIONS.values() for etiqueta in seccion]
all_labels_and_flags = labels + FLAGS

print(f'Categorias principales : {len(LABEL_SECTIONS)}')
print(f'Etiquetas de dano/seguro: {len(labels)}')
print(f'Flags transversales    : {len(FLAGS)}')
print()
for cat, etiquetas in LABEL_SECTIONS.items():
    print(f'{cat}:')
    for e in etiquetas:
        print(f'  - {e}')
print('\nFLAGS:')
for f in FLAGS:
    print(f'  - {f}')


Categorias principales : 4
Etiquetas de dano/seguro: 14
Flags transversales    : 3

SEGURO:
  - seguro
  - seguro_ironia_marcada
RACISMO_DISCRIMINACION:
  - racismo_etnico_explicito
  - racismo_linguistico
  - clasismo_racial
  - discriminacion_regional
  - racismo_encubierto
ACOSO:
  - misoginia_acoso_genero
  - homofobia_transfobia
  - acoso_personal
  - amenaza_directa
CONTENIDO_SEXUAL:
  - sexual_explicito
  - sexual_cosificacion
  - sexual_no_consensual

FLAGS:
  - ironia_ambigua
  - humor_encubridor
  - contexto_necesario


## 2. Frontend HTML externo

El archivo del frontend se mantiene fuera del notebook. Flujo recomendado:

1. Abrir `Cuadernos/frontend/etiquetado_humano.html`.
2. Verificar/actualizar el bloque embebido `embeddedChunks` dentro del HTML.
3. **Configurar anotador** en la parte superior: tipo (Humano / LLM), iniciales (máx. 3 letras) y modelo si aplica.
4. Etiquetar fragmentos presentados en **orden aleatorio** con la taxonomía peruana (multi-etiqueta + flags).
5. Exportar — el archivo se nombra automáticamente `<iniciales>_labeled_chunks.jsonl` y contiene solo chunks etiquetados.
6. Depositar en `datos/etiquetado/` para consolidación multi-anotador (sección 4).

Un mismo chunk puede ser anotado por humanos y LLMs en sesiones separadas.
La clave única por registro es `(chunk_id, annotator_id)`.
Las iniciales y metadatos del anotador persisten entre chunks y recargas del navegador.


In [14]:
from pathlib import Path

html_path = FRONTEND_DIR / 'etiquetado_humano.html'
if not html_path.exists():
    raise FileNotFoundError(f'No existe el frontend: {html_path}')

print('Frontend disponible en:', html_path)


Frontend disponible en: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\Cuadernos\frontend\etiquetado_humano.html


## 3. Archivos de contexto para clasificación con LLM

Para que un LLM comercial clasifique chunks automáticamente, carga estos tres archivos:

| Rol en el prompt | Archivo |
|---|---|
| **System message** (taxonomía + 7 pasos + 13 ejemplos) | `modelos/skills/clasificacion_moderacion_peru.md` |
| Referencia de etiquetas y fuentes | `datos/processed/taxonomia_moderacion.csv` |
| Chunks a clasificar | `datos/processed/chunks_para_etiquetar.jsonl` |

**Instrucción de uso:**
- Cargar el skill `.md` completo como `system message`.
- Enviar cada chunk como `user message`: `{"chunk_id": "...", "text": "..."}`.
- Solicitar respuesta en el JSON de salida definido en la sección 4.
- Guardar cada respuesta en `datos/etiquetado/<modelo>_labeled_chunks.jsonl` con `annotator_type="llm"`.

In [15]:
CONTEXT_FILES = {
    'system_prompt (skill)': ROOT / 'modelos' / 'skills' / 'clasificacion_moderacion_peru.md',
    'taxonomia_csv':         PROCESSED_DIR / 'taxonomia_moderacion.csv',
    'chunks_jsonl':          PROCESSED_DIR / 'chunks_para_etiquetar.jsonl',
    'frontend_html':         FRONTEND_DIR  / 'etiquetado_humano.html',
}

print('Archivos necesarios:')
for rol, path in CONTEXT_FILES.items():
    estado = '✓ existe' if path.exists() else '✗ no encontrado'
    print(f'  [{estado}]  {rol}')
    print(f'             {path}')


Archivos necesarios:
  [✓ existe]  system_prompt (skill)
             D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\modelos\skills\clasificacion_moderacion_peru.md
  [✓ existe]  taxonomia_csv
             D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\processed\taxonomia_moderacion.csv
  [✓ existe]  chunks_jsonl
             D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\processed\chunks_para_etiquetar.jsonl
  [✓ existe]  frontend_html
             D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\Cuadernos\frontend\etiquetado_humano.html


In [16]:
# ── Esquema de salida del etiquetado ─────────────────────────────────────────
# Clave única por registro: (chunk_id, annotator_id)
# El mismo chunk puede tener N filas, una por anotador (humano o LLM).

ANNOTATED_FIELDS = {
    # Del chunk original
    'chunk_id':        'str   — identificador único del chunk',
    'video_id':        'str',
    'channel_title':   'str',
    'start_seconds':   'float',
    'end_seconds':     'float',
    'text':            'str',
    'text_hash':       'str',
    # Etiquetas
    'labels':          'list  — subcategorías activas (multi-etiqueta)',
    'flags':           'list  — flags transversales activos',
    'needs_review':    'bool  — True si hay flags o sin etiqueta confirmada',
    'notes':           'str   — comentario libre del anotador',
    # Metadatos del anotador
    'annotator_type':  '"human" | "llm"',
    'annotator_id':    'str   — iniciales, máx. 3 caracteres (ej. "AKM", "G4O")',
    'annotator_model': 'str   — nombre del modelo LLM (ej. "gpt-4o"), null si humano',
    'skill_file':      'str   — skill .md usado, null si humano',
    'score_confianza': 'float — 0.0–1.0 para LLM, null para humano',
    'justificacion':   'str   — justificación textual del LLM, vacío para humano',
    # Auditoría
    'annotated_at':    'str   — ISO 8601 timestamp',
}

print('Esquema de registro de etiquetado:')
for campo, desc in ANNOTATED_FIELDS.items():
    print(f'  {campo:<20}  {desc}')


Esquema de registro de etiquetado:
  chunk_id              str   — identificador único del chunk
  video_id              str
  channel_title         str
  start_seconds         float
  end_seconds           float
  text                  str
  text_hash             str
  labels                list  — subcategorías activas (multi-etiqueta)
  flags                 list  — flags transversales activos
  needs_review          bool  — True si hay flags o sin etiqueta confirmada
  notes                 str   — comentario libre del anotador
  annotator_type        "human" | "llm"
  annotator_id          str   — iniciales, máx. 3 caracteres (ej. "AKM", "G4O")
  annotator_model       str   — nombre del modelo LLM (ej. "gpt-4o"), null si humano
  skill_file            str   — skill .md usado, null si humano
  score_confianza       float — 0.0–1.0 para LLM, null para humano
  justificacion         str   — justificación textual del LLM, vacío para humano
  annotated_at          str   — ISO 8601 time

## 4. Consolidación multi-anotador

El mismo chunk puede ser anotado por múltiples humanos y/o LLMs. El JSONL de salida tiene **una fila por `(chunk_id, annotator_id)`**.

| Escenario | Uso |
|---|---|
| Humano + LLM | El humano revisa y confirma/corrige la etiqueta del LLM |
| Múltiples humanos | Calcular Cohen's kappa por categoría antes del entrenamiento |
| Múltiples LLMs | Comparar acuerdo entre modelos; detectar sesgos |

`consolidar_anotaciones` produce el *gold standard* por mayoría de votos (>50%) entre todos los anotadores de cada chunk.

In [17]:
LABELED_FILE = LABELED_DIR / 'labeled_chunks.jsonl'


def _load_jsonl(path):
    if not path.exists():
        return []
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def save_annotation(record, path=LABELED_FILE):
    """Guarda o actualiza una anotación. Clave única: (chunk_id, annotator_id).
    Si ya existe un registro con la misma clave, lo reemplaza.
    """
    existing = _load_jsonl(path)
    key = (record.get('chunk_id'), record.get('annotator_id'))
    updated = [r for r in existing
               if (r.get('chunk_id'), r.get('annotator_id')) != key]
    updated.append(record)
    with open(path, 'w', encoding='utf-8') as f:
        for r in updated:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def consolidar_anotaciones(path=LABELED_FILE, min_anotadores=2):
    """Gold standard por mayoría de votos (>50%) entre anotadores del mismo chunk.

    Returns:
        DataFrame con una fila por chunk_id y etiquetas consensuadas.
        Columna 'n_anotadores' indica cuántos anotadores contribuyeron.
    """
    rows = _load_jsonl(path)
    if not rows:
        print('Sin anotaciones para consolidar.')
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    consolidado = []
    for chunk_id, group in df.groupby('chunk_id'):
        n = len(group)
        if n < min_anotadores:
            continue
        label_counts = Counter(itertools.chain.from_iterable(group['labels'].dropna()))
        flag_counts  = Counter(itertools.chain.from_iterable(group['flags'].dropna()))
        row = group.iloc[0].to_dict()
        row['labels']         = sorted([l for l, c in label_counts.items() if c / n > 0.5])
        row['flags']          = sorted([f for f, c in flag_counts.items()  if c / n > 0.5])
        row['n_anotadores']   = n
        row['annotator_type'] = 'consensus'
        row['annotator_id']   = 'CON'
        row['annotated_at']   = pd.Timestamp.now().isoformat()
        consolidado.append(row)

    return pd.DataFrame(consolidado)


def resumen_anotaciones(path=LABELED_FILE):
    """Muestra un resumen de las anotaciones disponibles por anotador."""
    rows = _load_jsonl(path)
    if not rows:
        print(f'Sin anotaciones en {path.name}')
        print(f'Deposita archivos *_labeled_chunks.jsonl en: {LABELED_DIR}')
        return
    df = pd.DataFrame(rows)
    print(f'Total de registros   : {len(df)}')
    print(f'Chunks únicos        : {df["chunk_id"].nunique()}')
    print(f'Anotadores únicos    : {df["annotator_id"].nunique()}')
    print()
    print(df.groupby(['annotator_type', 'annotator_id'])
            .size().rename('n_anotaciones').to_string())


resumen_anotaciones()


Sin anotaciones en labeled_chunks.jsonl
Deposita archivos *_labeled_chunks.jsonl en: D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado


In [19]:
import re
import json

def _load_chunks_for_embedding():
    jsonl_path = PROCESSED_DIR / 'chunks_para_etiquetar.jsonl'
    csv_path = PROCESSED_DIR / 'chunks_para_etiquetar.csv'

    if jsonl_path.exists():
        rows = _load_jsonl(jsonl_path)
        source = jsonl_path
    elif csv_path.exists():
        df = pd.read_csv(csv_path)
        rows = df.to_dict(orient='records')
        source = csv_path
    else:
        raise FileNotFoundError('No se encontró chunks_para_etiquetar.jsonl ni chunks_para_etiquetar.csv en datos/processed')

    keep_fields = [
        'chunk_id', 'video_id', 'channel_id', 'channel_title', 'video_title',
        'published_at', 'start_seconds', 'end_seconds', 'text', 'text_hash'
    ]
    cleaned = []
    for row in rows:
        out = {k: row.get(k) for k in keep_fields}
        out['labels'] = row.get('labels') or []
        out['flags'] = row.get('flags') or []
        out['needs_review'] = bool(row.get('needs_review', False))
        out['notes'] = row.get('notes') or ''
        cleaned.append(out)
    return cleaned, source


def embed_chunks_in_html(html_file=FRONTEND_DIR / 'etiquetado_humano.html'):
    rows, source = _load_chunks_for_embedding()
    json_payload = json.dumps(rows, ensure_ascii=False)

    html = html_file.read_text(encoding='utf-8')
    pattern = r"<script id='embeddedChunks' type='application/json'>[\s\S]*?</script>"
    replacement = f"<script id='embeddedChunks' type='application/json'>{json_payload}</script>"
    if not re.search(pattern, html):
        raise ValueError("No se encontró el bloque <script id='embeddedChunks'> en el HTML")

    html_updated = re.sub(pattern, replacement, html, count=1)
    html_file.write_text(html_updated, encoding='utf-8')

    print(f'Fuente de chunks : {source}')
    print(f'Chunks embebidos : {len(rows)}')
    print(f'HTML actualizado : {html_file}')


embed_chunks_in_html()

Fuente de chunks : D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\processed\chunks_para_etiquetar.jsonl
Chunks embebidos : 15150
HTML actualizado : D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\Cuadernos\frontend\etiquetado_humano.html


In [26]:
import ast
from pathlib import Path

LABEL_HELP = {
    'SEGURO': {
        'seguro': 'Sin infraccion: texto informativo, descriptivo o humor sin ataque a persona o grupo.',
        'seguro_ironia_marcada': 'Parodia/ironia cuyo blanco NO es un grupo humano.',
    },
    'RACISMO_DISCRIMINACION': {
        'racismo_etnico_explicito': 'Uso derogatorio de terminos etnicos (ej. serrano/cholo/negro/indio).',
        'racismo_linguistico': 'Burla de acento andino, motoseo u ortografia de migrantes.',
        'clasismo_racial': 'Inferiorizacion por clase con connotacion etnica (ej. huachafa/chusma/amixer).',
        'discriminacion_regional': 'Ataque por origen regional (Lima vs provincias, etc.).',
        'racismo_encubierto': 'Discriminacion disfrazada de criterio de cultura/educacion.',
    },
    'ACOSO': {
        'misoginia_acoso_genero': 'Insultos sexualizados, degradacion o ataque por genero.',
        'homofobia_transfobia': 'Insultos o amenazas contra personas LGBTQ+.',
        'acoso_personal': 'Ataque dirigido a persona identificable o doxeo.',
        'amenaza_directa': 'Expresion explicita de intencion de dano fisico/legal/economico.',
    },
    'CONTENIDO_SEXUAL': {
        'sexual_explicito': 'Descripcion grafica de actos sexuales sin proposito informativo.',
        'sexual_cosificacion': 'Sexualizacion/cosificacion de personas.',
        'sexual_no_consensual': 'Referencia a contenido sexual no consensual.',
    },
}

FLAG_HELP = {
    'ironia_ambigua': 'No se determina claramente si hay dano o parodia critica. Requiere revision.',
    'humor_encubridor': 'El humor funciona como cobertura para minimizar/normalizar el dano.',
    'contexto_necesario': 'El chunk aislado no alcanza; se necesita contexto adicional del video.',
}

def _coerce_list(value):
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass
    return []

def load_chunks_dataset(dataset_path=None):
    """Carga chunks desde .jsonl, .json o .csv para incrustarlos en el frontend."""
    if dataset_path is None:
        default_jsonl = PROCESSED_DIR / 'chunks_para_etiquetar.jsonl'
        default_csv = PROCESSED_DIR / 'chunks_para_etiquetar.csv'
        dataset_path = default_jsonl if default_jsonl.exists() else default_csv

    path = Path(dataset_path)
    if not path.exists():
        raise FileNotFoundError(f'No existe dataset: {path}')

    suffix = path.suffix.lower()
    if suffix == '.jsonl':
        rows = _load_jsonl(path)
    elif suffix == '.json':
        rows = json.loads(path.read_text(encoding='utf-8'))
        if not isinstance(rows, list):
            raise ValueError('El .json debe contener una lista de registros.')
    elif suffix == '.csv':
        rows = pd.read_csv(path).to_dict(orient='records')
    else:
        raise ValueError('Formato no soportado. Usa .jsonl, .json o .csv')

    keep_fields = [
        'chunk_id', 'video_id', 'channel_id', 'channel_title', 'video_title',
        'published_at', 'start_seconds', 'end_seconds', 'text', 'text_hash'
    ]
    cleaned = []
    for row in rows:
        out = {k: row.get(k) for k in keep_fields}
        out['labels'] = _coerce_list(row.get('labels'))
        out['flags'] = _coerce_list(row.get('flags'))
        out['needs_review'] = bool(row.get('needs_review', False))
        out['notes'] = row.get('notes') or ''
        cleaned.append(out)

    return cleaned, path

def write_labels_guide_html(output_path=FRONTEND_DIR / 'guia_etiquetas.html'):
    """Genera la segunda pagina con explicacion de etiquetas y flags del proyecto."""
    output_path = Path(output_path)
    cards = []
    for category, labels in LABEL_HELP.items():
        items = ''.join([f"<li><strong>{k}</strong>: {v}</li>" for k, v in labels.items()])
        cards.append(f"<section class='card'><h2>{category}</h2><ul>{items}</ul></section>")

    flag_items = ''.join([f"<li><strong>{k}</strong>: {v}</li>" for k, v in FLAG_HELP.items()])

    html = f"""<!doctype html>
<html lang='es'>
<head>
  <meta charset='utf-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1'>
  <title>Guia de etiquetas - Moderacion</title>
  <style>
    :root {{ --bg:#f6f7f9; --panel:#fff; --ink:#18202b; --muted:#5d6b7a; --line:#d7dde5; --brand:#b4232f; }}
    * {{ box-sizing: border-box; }}
    body {{ margin:0; font-family: Arial, Helvetica, sans-serif; background:var(--bg); color:var(--ink); }}
    header {{ background:var(--brand); color:#fff; padding:14px 22px; }}
    main {{ max-width:1200px; margin:0 auto; padding:18px; display:grid; gap:12px; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); }}
    .card {{ background:var(--panel); border:1px solid var(--line); border-radius:8px; padding:14px; }}
    h1 {{ margin:0; font-size:22px; }}
    h2 {{ margin:0 0 10px; font-size:16px; }}
    ul {{ margin:0; padding-left:18px; }}
    li {{ margin:6px 0; line-height:1.4; }}
    .note {{ max-width:1200px; margin:0 auto 14px; padding:0 18px; color:var(--muted); font-size:13px; }}
  </style>
</head>
<body>
  <header><h1>Guia de etiquetas del proyecto</h1></header>
  <p class='note'>Esta guia resume las etiquetas y flags definidos para el etiquetado humano/LLM en este proyecto.</p>
  <main>
    {''.join(cards)}
    <section class='card'>
      <h2>FLAGS TRANSVERSALES</h2>
      <ul>{flag_items}</ul>
    </section>
  </main>
</body>
</html>"""

    output_path.write_text(html, encoding='utf-8')
    return output_path

def build_frontend_html(rows):
    json_payload = json.dumps(rows, ensure_ascii=False)

    label_cards_html = []
    for category, labels in LABEL_HELP.items():
        items = ''.join([f"<li><strong>{k}</strong>: {v}</li>" for k, v in labels.items()])
        label_cards_html.append(f"<section class='guide-card'><h3>{category}</h3><ul>{items}</ul></section>")
    guide_labels_html = ''.join(label_cards_html)
    guide_flags_html = ''.join([f"<li><strong>{k}</strong>: {v}</li>" for k, v in FLAG_HELP.items()])

    html_template = """<!doctype html>
<html lang='es'>
<head>
  <meta charset='utf-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1'>
  <title>Etiquetado humano - Moderacion de contenido</title>
  <style>
    :root { --bg: #f6f7f9; --panel: #ffffff; --ink: #18202b; --muted: #5d6b7a; --line: #d7dde5; --brand: #b4232f; --ok: #0f7a45; --touch: 44px; }
    * { box-sizing: border-box; }
    body { margin: 0; font-family: Arial, Helvetica, sans-serif; background: var(--bg); color: var(--ink); }
    header { background: var(--brand); color: white; padding: 12px 16px; }
    main { max-width: 1320px; margin: 0 auto; padding: 14px; display: grid; grid-template-columns: 1fr 260px; gap: 14px; }
    section, aside { background: var(--panel); border: 1px solid var(--line); border-radius: 10px; padding: 14px; }
    h1 { font-size: 20px; margin: 0; }
    h2 { font-size: 16px; margin: 0 0 12px; }
    button { border: 1px solid var(--line); background: white; border-radius: 8px; padding: 10px 14px; cursor: pointer; min-height: var(--touch); font-size: 14px; }
    button.primary { background: var(--brand); color: white; border-color: var(--brand); }
    button.linklike { color: #1248a3; border-color: #bcd0ee; background: #f6f9ff; }
    input[type='text'], textarea { border: 1px solid var(--line); border-radius: 8px; font-family: inherit; }
    input[type='text'] { min-height: 40px; padding: 6px 10px; }
    textarea { width: 100%; min-height: 96px; resize: vertical; padding: 10px; }
    .toolbar { display: flex; gap: 8px; flex-wrap: wrap; align-items: center; margin-bottom: 10px; }
    .top-sticky { position: sticky; top: 0; z-index: 30; background: var(--panel); padding: 8px 0; border-bottom: 1px solid var(--line); }
    .meta { color: var(--muted); font-size: 13px; line-height: 1.35; margin-bottom: 10px; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
    .chunk { border: 1px solid var(--line); border-radius: 8px; padding: 14px; min-height: 220px; line-height: 1.55; background: #fbfcfd; }
    .labels { display: grid; grid-template-columns: 1fr; gap: 8px; }
    .label { border: 1px solid var(--line); border-radius: 6px; padding: 8px; display: flex; gap: 8px; align-items: flex-start; cursor: pointer; }
    .label input { margin-top: 2px; transform: scale(1.1); }
    .label-text { font-size: 13px; user-select: none; }
    .annotation-layout { display: grid; grid-template-columns: 1fr 290px 290px; gap: 12px; align-items: start; }
    .labels-panel { border: 1px solid var(--line); border-radius: 8px; padding: 10px; background: #fcfdff; max-height: 74vh; overflow: auto; }
    .chunk-center { min-width: 0; }
    .video-link { margin-bottom: 10px; font-size: 13px; }
    .video-link a { color: #1248a3; text-decoration: none; font-weight: bold; }
    .video-link a:hover { text-decoration: underline; }
    .quick-links { margin-top: 8px; display: flex; flex-wrap: wrap; gap: 8px; }
    .status { margin-top: 12px; font-size: 13px; color: var(--muted); }
    .ok { color: var(--ok); font-weight: bold; }
    .cat-seguro { background: #eaf8ee; border-color: #b8e2c3; }
    .cat-racismo { background: #fff1f1; border-color: #f0c2c2; }
    .cat-acoso { background: #fff7e8; border-color: #efd7a9; }
    .cat-sexual { background: #f1edff; border-color: #d2c5ff; }
    .cat-flag { background: #eaf4ff; border-color: #bcd8f5; }

    .guide-dialog { width: min(980px, 96vw); border: none; border-radius: 12px; padding: 0; }
    .guide-dialog::backdrop { background: rgba(0, 0, 0, 0.45); }
    .guide-shell { background: #fff; color: var(--ink); }
    .guide-head { display: flex; justify-content: space-between; align-items: center; gap: 8px; padding: 12px 14px; border-bottom: 1px solid var(--line); position: sticky; top: 0; background: #fff; }
    .guide-grid { display: grid; gap: 10px; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr)); padding: 12px; }
    .guide-card { border: 1px solid var(--line); border-radius: 8px; padding: 10px; background: #fbfcfd; }
    .guide-card h3 { margin: 0 0 8px; font-size: 14px; }
    .guide-card ul { margin: 0; padding-left: 18px; }
    .guide-card li { margin: 6px 0; font-size: 13px; line-height: 1.4; }

    @media (max-width: 1150px) {
      main { grid-template-columns: 1fr; }
      .annotation-layout { grid-template-columns: 1fr; }
      .labels-panel { max-height: none; }
    }

    @media (max-width: 768px) {
      header { position: sticky; top: 0; z-index: 40; padding: 10px 12px; }
      h1 { font-size: 17px; }
      main { padding: 10px; gap: 10px; }
      section, aside { padding: 10px; border-radius: 8px; }
      .toolbar { gap: 6px; }
      .top-sticky { top: 0; margin: -4px 0 8px; }
      .toolbar button { flex: 1 1 31%; min-width: 96px; }
      #annotatorId, #annotatorModel, #skillFile { width: 100% !important; }
      .meta { font-size: 12px; }
      .chunk { min-height: 180px; padding: 12px; }
      .label { padding: 10px; }
      .label-text { font-size: 14px; }
      aside { display: none; }
    }
  </style>
</head>
<body>
  <header>
    <h1>Etiquetado humano - Moderacion de contenido</h1>
    <div class='quick-links'>
      <button type='button' class='linklike' data-open-guide='1'>Guia de etiquetas y ejemplos</button>
    </div>
  </header>
  <main>
    <section>
      <div class='toolbar top-sticky'>
        <strong style='font-size:12px'>Anotador</strong>
        <label style='font-size:12px'><input type='radio' name='ann_type' value='human' checked> Humano</label>
        <label style='font-size:12px'><input type='radio' name='ann_type' value='llm'> LLM</label>
        <input id='annotatorId' type='text' maxlength='3' placeholder='Iniciales (max. 3)' style='width:140px;font-size:12px'>
        <input id='annotatorModel' type='text' placeholder='Modelo LLM (ej. gpt-4o)' style='display:none;width:190px;font-size:12px'>
        <input id='skillFile' type='text' value='clasificacion_moderacion_peru.md' style='display:none;width:220px;font-size:12px;color:var(--muted)'>
      </div>
      <div class='toolbar'>
        <button id='prevBtn'>Anterior</button>
        <button id='nextBtn' class='primary'>Siguiente</button>
        <button id='exportBtn'>Exportar JSONL</button>
        <button id='openGuideBtn' class='linklike' type='button' data-open-guide='1'>Guia</button>
      </div>
      <div id='meta' class='meta'>Cargando chunks embebidos...</div>
      <div id='videoSource' class='video-link'>Video original: cargando...</div>
      <div class='annotation-layout'>
        <div class='chunk-center'>
          <div id='chunkText' class='chunk'>Preparando interfaz...</div>
          <h2 style='margin-top:14px'>Notas</h2>
          <textarea id='notes' placeholder='Comentario breve del anotador'></textarea>
          <div id='status' class='status'></div>
        </div>
        <div id='labelsLeft' class='labels-panel labels'></div>
        <div id='labelsRight' class='labels-panel labels'></div>
      </div>
    </section>
    <aside>
      <h2>Ayuda rapida</h2>
      <p style='font-size:13px;color:var(--muted);line-height:1.45'>Todo esta en un solo archivo HTML: chunks, interfaz y guia de etiquetas.</p>
      <button type='button' data-open-guide='1' class='linklike'>Abrir guia de etiquetas</button>
    </aside>
  </main>

  <dialog id='guideDialog' class='guide-dialog'>
    <div class='guide-shell'>
      <div class='guide-head'>
        <h2 style='margin:0'>Guia de etiquetas del proyecto</h2>
        <button type='button' id='closeGuideBtn'>Cerrar</button>
      </div>
      <div class='guide-grid'>
        __GUIDE_LABEL_CARDS__
        <section class='guide-card'>
          <h3>FLAGS TRANSVERSALES</h3>
          <ul>__GUIDE_FLAG_ITEMS__</ul>
        </section>
      </div>
    </div>
  </dialog>

  <script id='embeddedChunks' type='application/json'>__EMBEDDED_JSON__</script>
  <script>
    const LABEL_SECTIONS = [
      ['SEGURO', ['seguro', 'seguro_ironia_marcada']],
      ['RACISMO / DISCRIMINACION', ['racismo_etnico_explicito','racismo_linguistico','clasismo_racial','discriminacion_regional','racismo_encubierto']],
      ['ACOSO', ['misoginia_acoso_genero','homofobia_transfobia','acoso_personal','amenaza_directa']],
      ['CONTENIDO SEXUAL', ['sexual_explicito','sexual_cosificacion','sexual_no_consensual']],
    ];
    const SECTION_CLASS = {
      'SEGURO': 'cat-seguro',
      'RACISMO / DISCRIMINACION': 'cat-racismo',
      'ACOSO': 'cat-acoso',
      'CONTENIDO SEXUAL': 'cat-sexual',
    };
    const SAFE_LABELS = ['seguro', 'seguro_ironia_marcada'];
    const FLAGS = ['ironia_ambigua', 'humor_encubridor', 'contexto_necesario'];
    const DISPLAY_LABELS = {
      acoso_personal: 'Acoso personal/lenguaje obceno',
      amenaza_directa: 'Amenza directa/Violencia'
    };
    const state = { rows: [], order: [], index: 0 };
    const meta = document.getElementById('meta');
    const chunkText = document.getElementById('chunkText');
    const labelsLeft = document.getElementById('labelsLeft');
    const labelsRight = document.getElementById('labelsRight');
    const notes = document.getElementById('notes');
    const status = document.getElementById('status');
    const videoSource = document.getElementById('videoSource');
    const STORAGE_KEY = 'moderacion.annotator.v1';
    const guideDialog = document.getElementById('guideDialog');

    function shuffleInPlace(arr) {
      for (let i = arr.length - 1; i > 0; i -= 1) {
        const j = Math.floor(Math.random() * (i + 1));
        [arr[i], arr[j]] = [arr[j], arr[i]];
      }
      return arr;
    }

    function currentRow() {
      if (!state.rows.length || !state.order.length) return null;
      return state.rows[state.order[state.index]] || null;
    }

    function formatLabelName(label) {
      return DISPLAY_LABELS[label] || label.replaceAll('_', ' ');
    }

    function createSectionBlock(parent, sectionName, labels) {
      const hdr = document.createElement('div');
      hdr.style.cssText = 'font-size:10px;font-weight:bold;color:var(--muted);text-transform:uppercase;margin:6px 0 2px;letter-spacing:.05em';
      hdr.textContent = sectionName;
      parent.appendChild(hdr);
      const sectionClass = SECTION_CLASS[sectionName] || '';
      labels.forEach(label => {
        const wrap = document.createElement('label');
        wrap.className = `label ${sectionClass}`;
        wrap.innerHTML = `<input class='label-checkbox' type='checkbox' value='${label}'> <span class='label-text'>${formatLabelName(label)}</span>`;
        parent.appendChild(wrap);
      });
    }

    function renderLabels() {
      labelsLeft.innerHTML = '';
      labelsRight.innerHTML = '';
      const leftSections = LABEL_SECTIONS.slice(0, 2);
      const rightSections = LABEL_SECTIONS.slice(2);
      leftSections.forEach(([name, labels]) => createSectionBlock(labelsLeft, name, labels));
      rightSections.forEach(([name, labels]) => createSectionBlock(labelsRight, name, labels));

      const fhdr = document.createElement('div');
      fhdr.style.cssText = 'font-size:10px;font-weight:bold;color:var(--muted);text-transform:uppercase;margin:8px 0 2px;letter-spacing:.05em;border-top:1px solid var(--line);padding-top:6px';
      fhdr.textContent = 'FLAGS TRANSVERSALES';
      labelsRight.appendChild(fhdr);
      FLAGS.forEach(flag => {
        const wrap = document.createElement('label');
        wrap.className = 'label cat-flag';
        wrap.innerHTML = `<input class='label-checkbox' type='checkbox' value='${flag}'> <span class='label-text'><em>${formatLabelName(flag)}</em></span>`;
        labelsRight.appendChild(wrap);
      });

      const labelsRoot = document.querySelector('.annotation-layout');
      labelsRoot.addEventListener('click', event => {
        if (!event.target.classList.contains('label-text')) return;
        event.preventDefault();
        const label = event.target.closest('label');
        if (!label) return;
        const checkbox = label.querySelector('.label-checkbox');
        if (!checkbox) return;
        checkbox.checked = !checkbox.checked;
        checkbox.dispatchEvent(new Event('change', { bubbles: true }));
      });

      labelsRoot.addEventListener('change', event => {
        if (!event.target.classList.contains('label-checkbox')) return;
        const val = event.target.value;
        if (SAFE_LABELS.includes(val) && event.target.checked) {
          document.querySelectorAll('.label-checkbox').forEach(x => {
            if (!SAFE_LABELS.includes(x.value) && !FLAGS.includes(x.value)) x.checked = false;
          });
        }
        if (!SAFE_LABELS.includes(val) && !FLAGS.includes(val) && event.target.checked) {
          SAFE_LABELS.forEach(s => {
            const el = document.querySelector(`.label-checkbox[value='${s}']`);
            if (el) el.checked = false;
          });
        }
        saveCurrent();
      });
    }

    function parseRows(text) {
      const trimmed = text.trim();
      if (!trimmed) return [];
      if (trimmed.startsWith('[')) return JSON.parse(trimmed);
      return trimmed.split(String.fromCharCode(10)).map(line => line.replace(String.fromCharCode(13), '')).filter(Boolean).map(line => JSON.parse(line));
    }

    function selectedLabels() {
      return [...document.querySelectorAll('.label-checkbox:checked')].map(x => x.value);
    }

    function formatTimestamp(totalSeconds) {
      const sec = Math.max(0, Math.floor(Number(totalSeconds) || 0));
      const h = Math.floor(sec / 3600);
      const m = Math.floor((sec % 3600) / 60);
      const s = sec % 60;
      if (h > 0) return `${h}:${String(m).padStart(2, '0')}:${String(s).padStart(2, '0')}`;
      return `${String(m).padStart(2, '0')}:${String(s).padStart(2, '0')}`;
    }

    function getAnnotatorMeta() {
      const type = document.querySelector('input[name=ann_type]:checked')?.value || 'human';
      const id = (document.getElementById('annotatorId')?.value || '').trim().toUpperCase().slice(0, 3);
      const model = type === 'llm' ? (document.getElementById('annotatorModel')?.value.trim() || null) : null;
      const skill = type === 'llm' ? (document.getElementById('skillFile')?.value.trim() || null) : null;
      return { annotator_type: type, annotator_id: id, annotator_model: model, skill_file: skill };
    }

    function persistAnnotatorMeta() {
      localStorage.setItem(STORAGE_KEY, JSON.stringify(getAnnotatorMeta()));
    }

    function restoreAnnotatorMeta() {
      try {
        const raw = localStorage.getItem(STORAGE_KEY);
        if (!raw) return;
        const ann = JSON.parse(raw);
        const radio = document.querySelector(`input[name=ann_type][value='${ann.annotator_type || 'human'}']`);
        if (radio) radio.checked = true;
        if (ann.annotator_id) document.getElementById('annotatorId').value = ann.annotator_id;
        if (ann.annotator_model) document.getElementById('annotatorModel').value = ann.annotator_model;
        if (ann.skill_file) document.getElementById('skillFile').value = ann.skill_file;
      } catch (e) {}
    }

    function updateAnnotatorFieldVisibility() {
      const llm = document.querySelector('input[name=ann_type]:checked')?.value === 'llm';
      document.getElementById('annotatorModel').style.display = llm ? 'inline-block' : 'none';
      document.getElementById('skillFile').style.display = llm ? 'inline-block' : 'none';
    }

    function isLabeled(row) {
      const nLabels = (row.labels || []).length;
      const nFlags = (row.flags || []).length;
      const hasNotes = Boolean((row.notes || '').trim());
      return nLabels > 0 || nFlags > 0 || hasNotes;
    }

    function saveCurrent() {
      const row = currentRow();
      if (!row) return;
      const all = selectedLabels();
      const ann = getAnnotatorMeta();
      row.labels = all.filter(l => !FLAGS.includes(l));
      row.flags = all.filter(l => FLAGS.includes(l));
      row.notes = notes.value;
      row.needs_review = row.labels.length === 0 || row.flags.length > 0;
      row.annotated_at = new Date().toISOString();
      row.annotator_type = ann.annotator_type;
      row.annotator_id = ann.annotator_id;
      row.annotator_model = ann.annotator_model;
      row.skill_file = ann.skill_file;
      row.score_confianza = row.score_confianza ?? null;
      row.justificacion = row.justificacion ?? '';
      persistAnnotatorMeta();
    }

    function buildMetaLine(row) {
      const idx = state.index + 1;
      const total = state.rows.length;
      const channel = row.channel_title || '';
      const title = row.video_title || row.title || '';
      const start = Number(row.start_seconds || 0).toFixed(1);
      const end = Number(row.end_seconds || 0).toFixed(1);
      if (window.innerWidth <= 768) {
        return `${idx}/${total} | ${channel || 'Canal'} | ${start}s-${end}s`;
      }
      return `${idx}/${total} (orden aleatorio) | ${channel} | ${title} | ${start}s - ${end}s`;
    }

    function render() {
      const row = currentRow();
      if (!row) {
        meta.textContent = 'Sin chunks embebidos.';
        chunkText.textContent = 'Agrega JSON en embeddedChunks.';
        videoSource.textContent = 'Video original: no disponible';
        return;
      }
      meta.textContent = buildMetaLine(row);
      const start = Number(row.start_seconds || 0);
      const videoId = (row.video_id || '').trim();
      if (videoId) {
        const ts = formatTimestamp(start);
        const url = `https://www.youtube.com/watch?v=${videoId}&t=${Math.floor(start)}s`;
        videoSource.innerHTML = `Video original en YouTube: <a href='${url}' target='_blank' rel='noopener'>${ts}</a>`;
      } else {
        videoSource.textContent = 'Video original: no disponible para este chunk';
      }
      chunkText.textContent = row.text || '';
      const active = [...(row.labels || []), ...(row.flags || [])];
      document.querySelectorAll('.label-checkbox').forEach(x => x.checked = active.includes(x.value));
      notes.value = row.notes || '';
      updateAnnotatorFieldVisibility();
      const ann = getAnnotatorMeta();
      const etiquetados = state.rows.filter(isLabeled).length;
      const icono = ann.annotator_type === 'llm' ? '🤖' : '👤';
      status.innerHTML = `<span class='ok'>${etiquetados}</span> etiquetados de ${state.rows.length} | ${icono} <strong>${ann.annotator_id || '—'}</strong>`;
    }

    function loadEmbeddedChunks() {
      const source = document.getElementById('embeddedChunks');
      if (!source) return [];
      const text = source.textContent || '[]';
      const parsed = parseRows(text);
      return Array.isArray(parsed) ? parsed : [];
    }

    function bindGuideDialog() {
      if (!guideDialog) return;
      document.querySelectorAll('[data-open-guide="1"]').forEach(btn => {
        btn.addEventListener('click', () => {
          if (typeof guideDialog.showModal === 'function') {
            guideDialog.showModal();
          }
        });
      });
      const closeBtn = document.getElementById('closeGuideBtn');
      if (closeBtn) closeBtn.addEventListener('click', () => guideDialog.close());
      guideDialog.addEventListener('click', (event) => {
        if (event.target === guideDialog) guideDialog.close();
      });
    }

    document.getElementById('prevBtn').addEventListener('click', () => { saveCurrent(); state.index = Math.max(0, state.index - 1); render(); });
    document.getElementById('nextBtn').addEventListener('click', () => { saveCurrent(); state.index = Math.min(state.rows.length - 1, state.index + 1); render(); });
    notes.addEventListener('input', saveCurrent);
    document.getElementById('exportBtn').addEventListener('click', () => {
      saveCurrent();
      const ann = getAnnotatorMeta();
      const prefix = ann.annotator_id ? ann.annotator_id.toLowerCase() + '_' : '';
      const labeledRows = state.rows.filter(isLabeled);
      const out = labeledRows.map(row => JSON.stringify(row)).join(String.fromCharCode(10));
      const blob = new Blob([out], { type: 'application/jsonl;charset=utf-8' });
      const a = document.createElement('a');
      a.href = URL.createObjectURL(blob);
      a.download = `${prefix}labeled_chunks.jsonl`;
      a.click();
      URL.revokeObjectURL(a.href);
    });

    window.addEventListener('resize', () => {
      if (currentRow()) meta.textContent = buildMetaLine(currentRow());
    });

    document.querySelectorAll('input[name=ann_type]').forEach(r => {
      r.addEventListener('change', () => { updateAnnotatorFieldVisibility(); saveCurrent(); });
    });
    document.getElementById('annotatorId').addEventListener('input', saveCurrent);
    document.getElementById('annotatorModel').addEventListener('input', saveCurrent);
    document.getElementById('skillFile').addEventListener('input', saveCurrent);

    restoreAnnotatorMeta();
    updateAnnotatorFieldVisibility();
    state.rows = loadEmbeddedChunks();
    state.order = shuffleInPlace(state.rows.map((_, i) => i));
    state.index = 0;
    renderLabels();
    bindGuideDialog();
    render();
  </script>
</body>
</html>"""

    return (
        html_template
        .replace('__EMBEDDED_JSON__', json_payload)
        .replace('__GUIDE_LABEL_CARDS__', guide_labels_html)
        .replace('__GUIDE_FLAG_ITEMS__', guide_flags_html)
    )


def recreate_frontend_with_dataset(dataset_path=None,
                                  html_file=FRONTEND_DIR / 'etiquetado_humano.html',
                                  guide_file=FRONTEND_DIR / 'guia_etiquetas.html',
                                  export_standalone_guide=False):
    """Recrea recursos del frontend para cualquier set de datos de chunks.

    - Reescribe etiquetado_humano.html completo con chunks + guia embebida en un solo archivo.
    - Opcionalmente genera guia_etiquetas.html externa si export_standalone_guide=True.
    """
    rows, source = load_chunks_dataset(dataset_path)

    html_file = Path(html_file)
    html_file.write_text(build_frontend_html(rows), encoding='utf-8')

    if export_standalone_guide:
        write_labels_guide_html(guide_file)

    print(f'Dataset fuente     : {source}')
    print(f'Chunks incrustados : {len(rows)}')
    print(f'Frontend HTML      : {html_file}')
    if export_standalone_guide:
        print(f'Guia etiquetas     : {guide_file}')
    else:
        print('Guia embebida      : SI (mismo archivo HTML)')


# Uso por defecto (dataset procesado actual):
recreate_frontend_with_dataset()

Dataset fuente     : D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\processed\chunks_para_etiquetar.jsonl
Chunks incrustados : 15150
Frontend HTML      : D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\Cuadernos\frontend\etiquetado_humano.html
Guia embebida      : SI (mismo archivo HTML)
